In [1]:
import datetime
import logging  # 追加：ログ制御用
import warnings  # 追加：警告非表示用
import pandas as pd
import yfinance as yf
from IPython.display import display, HTML

# 警告メッセージ（FutureWarningなど）を非表示にする
warnings.filterwarnings("ignore")

# yfinanceの内部エラー出力を非表示にする
yf_logger = logging.getLogger('yfinance')
yf_logger.setLevel(logging.CRITICAL)

# Define the lookback periods (in business days) for performance calculation.
LOOKBACK_DAYS = {
    "1 Day (%)": -2,
    "3 Days (%)": -4,
    "1 Week (%)": -6,
    "1 Month (%)": -21,
    "6 Months (%)": -126,
    "1 Year (%)": -252,
}

# =====================================================================
# DATA PROCESSING FUNCTION (Supports Market Segmentation)
# =====================================================================
def show_market_performances(market_dict):
    today = datetime.date.today()
    start_date = today - datetime.timedelta(days=60)
    
    # Format numerical values to 2 decimal places
    format_config = {"Latest Price": "{:,.2f}"}
    for column_name in LOOKBACK_DAYS.keys():
        format_config[column_name] = "{:+.2f}%"
        
    # ハイライト用の条件関数
    def highlight_extreme_values(val):
        if pd.isna(val):
            return ''
        if val >= 10:
            return 'background-color: #ffcccc; color: #cc0000; font-weight: bold;'
        elif val <= -10:
            return 'background-color: #cce5ff; color: #004085; font-weight: bold;'
        return ''
        
    # Process each market section
    for market_name, ticker_dict in market_dict.items():
        records = []
        
        # Display market header
        display(HTML(f"<h3 style='margin-top: 20px; color: #333;'>■ {market_name} Market</h3>"))
        
        for name, ticker in ticker_dict.items():
            try:
                # 【修正】 ignore_tz=True を追加してタイムゾーン警告を抑制
                df = yf.download(ticker, start=start_date, end=today, progress=False, ignore_tz=True)
                
                if df.empty:
                    # 元のコードにあったWarning表示も邪魔ならコメントアウトしてください
                    # print(f"[Warning] No data found for: {name} ({ticker})")
                    continue
                    
                prices = df["Close"].squeeze()
                latest_price = prices.iloc[-1]
                
                row = {
                    "Name": name,
                    "Ticker": ticker,
                    "Latest Price": latest_price
                }
                
                for label, index in LOOKBACK_DAYS.items():
                    if len(prices) >= abs(index):
                        past_price = prices.iloc[index]
                    else:
                        past_price = prices.iloc[0]
                    
                    row[label] = ((latest_price - past_price) / past_price) * 100
                    
                records.append(row)
                
            except Exception as e:
                # エラー時のprintも非表示にしたい場合はコメントアウトしてください
                pass
        
        if records:
            df_performance = pd.DataFrame(records)
            
            # .map を使用
            styled_df = (df_performance.style
                         .format(format_config)
                         .map(highlight_extreme_values, subset=list(LOOKBACK_DAYS.keys()))
                         .hide(axis="index"))
            
            display(styled_df)
        else:
            print("No data available for this market.")

In [2]:
# =====================================================================
# CONFIGURATION & EXECUTION (Segmented by Market)
# =====================================================================
MARKET_TICKERS = {
    "United States": {
        "S&P 500": "^GSPC",
        "NASDAQ Composite": "^IXIC",
        "Apple": "AAPL",
        "NVIDIA": "NVDA",
        "Microsoft": "MSFT",
    },
    "Japan": {
        "Nikkei 225": "^N225",
        "Toyota Motor": "7203.T",
        "Sony Group": "6758.T",
        "Mitsubishi UFJ Financial": "8306.T",
    },
    "Singapore": {
        "STI Index": "^STI",
        "DBS Group": "D05.SI",
        "UOB": "U11.SI",
        "Singtel": "Z74.SI",
        "CapitaLand Ascendas REIT": "A17U.SI",
    },
    "Japan Stock": {
        "小松製作所 (6301)": "6301.T",
        "ニデック (6594)": "6594.T",
        "未来工業 (7931)": "7931.T",
        "東部ネットワーク (9036)": "9036.T",
        "ニトリHD (9843)": "9843.T",
        "MRK HD (9980)": "9980.T",
    },
    "US Stock (ETF)": {
        "iシェアーズ コア米国総合債券ETF (AGG)": "AGG"
    }
}

# Run and display the segmented tables
show_market_performances(MARKET_TICKERS)

Name,Ticker,Latest Price,1 Day (%),3 Days (%),1 Week (%),1 Month (%),6 Months (%),1 Year (%)
S&P 500,^GSPC,"7,483.23",-0.22%,+1.76%,+1.70%,-1.66%,+3.92%,+3.92%
NASDAQ Composite,^IXIC,"26,040.03",-0.66%,+2.93%,+2.21%,-3.89%,+3.88%,+3.88%
Apple,AAPL,294.38,+1.73%,+3.74%,+0.44%,-6.61%,+6.44%,+6.44%
NVIDIA,NVDA,197.58,-1.25%,+2.62%,-0.71%,-11.22%,-0.34%,-0.34%
Microsoft,MSFT,384.28,+3.02%,+3.03%,+5.15%,-12.92%,-6.89%,-6.89%


Name,Ticker,Latest Price,1 Day (%),3 Days (%),1 Week (%),1 Month (%),6 Months (%),1 Year (%)
Nikkei 225,^N225,"70,474.96",+0.59%,+1.61%,+1.88%,+3.03%,+12.16%,+12.16%
Toyota Motor,7203.T,"2,724.50",-0.02%,-1.57%,+1.43%,-5.43%,-8.51%,-8.51%
Sony Group,6758.T,"3,250.00",-0.91%,+1.59%,+0.00%,-10.20%,+3.83%,+3.83%
Mitsubishi UFJ Financial,8306.T,"3,247.00",+1.25%,+0.06%,+1.31%,+3.51%,+13.41%,+13.41%


Name,Ticker,Latest Price,1 Day (%),3 Days (%),1 Week (%),1 Month (%),6 Months (%),1 Year (%)
STI Index,^STI,"5,161.50",-0.18%,-0.58%,-1.04%,+0.45%,+4.82%,+4.82%
DBS Group,D05.SI,65.43,+0.05%,+0.00%,-1.19%,+0.58%,+13.22%,+13.22%
UOB,U11.SI,39.65,-0.28%,-0.38%,-0.63%,+2.19%,+9.62%,+9.62%
Singtel,Z74.SI,4.44,+0.68%,+0.23%,+0.68%,+2.07%,-4.93%,-4.93%
CapitaLand Ascendas REIT,A17U.SI,2.46,-1.20%,-3.15%,-2.77%,-1.60%,-1.99%,-1.99%


Name,Ticker,Latest Price,1 Day (%),3 Days (%),1 Week (%),1 Month (%),6 Months (%),1 Year (%)
小松製作所 (6301),6301.T,"6,259.00",-0.37%,-1.11%,-3.43%,-11.99%,-4.81%,-4.81%
ニデック (6594),6594.T,"2,747.00",+3.86%,+8.02%,-0.11%,-3.92%,+5.94%,+5.94%
未来工業 (7931),7931.T,"3,085.00",+0.00%,+1.48%,+2.49%,+2.15%,-0.16%,-0.16%
東部ネットワーク (9036),9036.T,"1,258.00",+1.45%,+1.70%,+2.36%,-3.97%,+5.10%,+5.10%
ニトリHD (9843),9843.T,"2,310.00",-4.33%,-5.33%,-2.51%,-12.78%,+2.60%,+2.60%
MRK HD (9980),9980.T,93.00,+1.09%,+2.20%,+2.20%,-6.06%,-5.10%,-5.10%


Name,Ticker,Latest Price,1 Day (%),3 Days (%),1 Week (%),1 Month (%),6 Months (%),1 Year (%)
iシェアーズ コア米国総合債券ETF (AGG),AGG,98.50,-0.48%,-0.85%,-0.70%,-0.21%,+0.24%,+0.24%


In [7]:
MARKET_TICKERS = {
    "Singapore Healthcare": {
        # --- 病院経営・総合クリニック ---
        "Raffles Medical Group (ラッフルズ医療)": "BSL.SI",  # 【修正】R01.SI から変更
        "IHH Healthcare (マウント・エリザベス等経営)": "Q0F.SI",  # 【修正】I01.SI から変更
        "Thomson Medical Group (トムソン医療)": "A50.SI",
        
        # --- 医療不動産（REIT / 病院・介護施設） ---
        "Parkway Life REIT (パークウェイ・ライフ)": "C2PU.SI",
        "First REIT (ファースト・リート)": "AW9U.SI",
        
        # --- 専門医療・歯科クリニックチェーン ---
        "Q&M Dental Group (歯科大手チェーン)": "QC7.SI",
        "Alliance Healthcare Group (総合診療・クリニック)": "MIJ.SI",  # 【修正】1D8から変更
        "Medinex Limited (医療サポート・クリニック運営)": "OTX.SI",
        
        # --- 医療用消耗品・製薬 ---
        "Medtecs International (医療用消耗品)": "546.SI",
        "Hyphens Pharma (医薬品・皮膚科製品)": "1J5.SI",
        "Riverstone": "AP4.SI"
    }
}
show_market_performances(MARKET_TICKERS)

Name,Ticker,Latest Price,1 Day (%),3 Days (%),1 Week (%),1 Month (%),6 Months (%),1 Year (%)
Raffles Medical Group (ラッフルズ医療),BSL.SI,0.91,-0.55%,-1.09%,-1.62%,-3.70%,-6.16%,-6.16%
IHH Healthcare (マウント・エリザベス等経営),Q0F.SI,2.74,+0.74%,+0.74%,+1.11%,-2.14%,-2.49%,-2.49%
Thomson Medical Group (トムソン医療),A50.SI,0.05,+1.92%,+0.00%,+0.00%,-5.36%,-7.02%,-7.02%
Parkway Life REIT (パークウェイ・ライフ),C2PU.SI,4.04,-0.25%,-1.22%,-0.74%,+2.54%,+0.50%,+0.50%
First REIT (ファースト・リート),AW9U.SI,0.22,-2.17%,+2.27%,+0.00%,-4.26%,-6.25%,-6.25%
Q&M Dental Group (歯科大手チェーン),QC7.SI,0.55,+0.00%,-1.79%,-3.51%,-9.09%,-5.17%,-5.17%
Alliance Healthcare Group (総合診療・クリニック),MIJ.SI,0.16,+0.00%,+2.52%,+2.52%,-0.61%,+0.00%,+0.00%
Medinex Limited (医療サポート・クリニック運営),OTX.SI,0.22,-2.22%,-2.22%,-2.22%,-12.00%,-4.35%,-4.35%
Medtecs International (医療用消耗品),546.SI,0.12,+0.00%,-0.85%,-0.85%,-6.45%,-2.52%,-2.52%
Hyphens Pharma (医薬品・皮膚科製品),1J5.SI,0.35,+1.45%,+0.00%,+2.94%,+7.69%,+2.94%,+2.94%


In [4]:
MARKET_TICKERS = {
"Singapore Food & Agribusiness": {
        # --- 国際的アグリビジネス（巨大穀物・商社） ---
        "Wilmar International (ウィルマー・世界最大級のパーム油)": "F34.SI",
        "Olam Group (オラム・ココア、コーヒー、ナッツ等大手)": "VC2.SI",
        "Olam Agri Holdings (オラムのアグリ部門 ※上場状況による)": "OAG.SI",
        
        # --- パーム油・農園経営（プランテーション） ---
        "Golden Agri-Resources (ゴールデン・アグリ・パーム油大手)": "E5H.SI",
        "First Resources (ファースト・リソーシズ・パーム油生産)": "EB5.SI",
        "Bumitama Agri (ブミタマ・アグリ・インドネシア農園)": "P8D.SI",
        
        # --- 食品製造・流通・飲料 ---
        "Fraser and Neave (F&N・飲料・乳製品大手)": "F99.SI",
        "Japfa (ジャプファ・乳肉製品、畜産大手)": "UD2.SI",
        "Delfi Limited (デルフィ・チョコレート製造・流通)": "P34.SI",
        "Sheng Siong Group (シェンシオン・食品スーパー大手)": "OV8.SI"
    }
}
show_market_performances(MARKET_TICKERS)

Name,Ticker,Latest Price,1 Day (%),3 Days (%),1 Week (%),1 Month (%),6 Months (%),1 Year (%)
Wilmar International (ウィルマー・世界最大級のパーム油),F34.SI,3.63,+0.55%,-0.82%,-1.89%,+4.31%,-2.42%,-2.42%
Olam Group (オラム・ココア、コーヒー、ナッツ等大手),VC2.SI,1.20,-0.83%,-0.83%,-4.00%,-5.51%,+1.69%,+1.69%
Golden Agri-Resources (ゴールデン・アグリ・パーム油大手),E5H.SI,0.26,-1.85%,-1.85%,+0.00%,-5.36%,-16.18%,-16.18%
First Resources (ファースト・リソーシズ・パーム油生産),EB5.SI,3.08,-0.96%,-1.28%,-4.05%,+14.07%,-11.31%,-11.31%
Fraser and Neave (F&N・飲料・乳製品大手),F99.SI,1.43,+0.00%,+0.70%,+0.70%,-1.38%,+1.05%,+1.05%
Delfi Limited (デルフィ・チョコレート製造・流通),P34.SI,0.89,-0.56%,+0.56%,-1.65%,-8.67%,-16.95%,-16.95%
Sheng Siong Group (シェンシオン・食品スーパー大手),OV8.SI,3.19,-1.24%,-0.93%,-0.93%,+3.57%,+4.93%,+4.93%
